# 01 · Data analysis

A first look at Hospitals A and B of the PhysioNet/CinC 2019 sepsis data, before any model is trained: class balance, missingness, feature distributions, how the features behave in the hours before sepsis onset, and where the two hospitals differ.

- **Features:** 8 vital signs, 26 laboratory values and 6 demographic/administrative variables (the lists are in `sepsis.py`). Binary variables (Gender, Unit1, Unit2) are drawn as bars, all others as boxplots.
- **Sepsis onset** = first positive `SepsisLabel` + 6 hours, because the label switches on 6 hours before onset.
- The analyses over time use the 34 vital/lab measurements. Age, Gender, Unit1, Unit2 and HospAdmTime do not change during a stay, and ICULOS is the time axis itself.
- A panel that says *no data* means the feature is completely missing for that group (for example EtCO2 in Hospital A).

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil
import sepsis as s     # shared code: loading and the feature lists

VITALS, LABS, DEMOGRAPHICS = s.VITALS, s.LABS, s.DEMOGRAPHICS
CLINICAL, ALL_FEATURES = s.CLINICAL, s.ALL_FEATURES

#Read every .psv file ONCE. All cells below reuse these two tables instead of re-reading 40,000 files.
df_A = s.load_hospital("A")
df_B = s.load_hospital("B")
HOSPITALS = {"Hospital A": df_A, "Hospital B": df_B}

OSError: [Errno 22] Invalid argument: 'C:\\Users\\bJawh\\PycharmProjects\\Intro2ML_Project\\sepsis_dataset\\training_setB\\p109799.psv'

In [ ]:
def add_sepsis_columns(data):
    '''Add, to every row, whether the patient is septic and how far this hour is from onset.'''
    first_positive = data.loc[data["SepsisLabel"] == 1].groupby("PatientID")["Hour"].min()

    data["Septic"] = data["PatientID"].isin(first_positive.index)
    data["OnsetHour"] = data["PatientID"].map(first_positive) + 6    # NaN for patients who are never septic
    data["HoursToOnset"] = data["Hour"] - data["OnsetHour"]        # 0 = onset, -6 = first positive label
    data["PatientStatus"] = pd.Categorical(np.where(data["Septic"], "Septic", "Non-septic"),
                                           categories=["Non-septic", "Septic"])

    # A septic patient's hours are split in two: before onset (what we want to predict from) and after.
    phase = np.where(~data["Septic"], "Non-septic",
                     np.where(data["HoursToOnset"] >= 0, "Sepsis", "Pre-sepsis"))
    data["Phase"] = pd.Categorical(phase, categories=["Non-septic", "Pre-sepsis", "Sepsis"])


for name, df in HOSPITALS.items():
    add_sepsis_columns(df)
    print(f"{name}: {df['PatientID'].nunique():,} patients, {len(df):,} hourly rows")

In [ ]:
# Plotting helpers used by the analyses below
COLORS = {"Hospital A": "#4c78a8", "Hospital B": "#f58518",
          "Non-septic": "#9ecae1", "Septic": "#e6550d",
          "Pre-sepsis": "#fdae6b", "Sepsis": "#e6550d"}
BINARY = ["Gender", "Unit1", "Unit2"]      # 0/1 variables: a boxplot of 0s and 1s says nothing, so bars

FEATURE_GROUPS = {"Vital signs": VITALS, "Laboratory values": LABS, "Demographic / administrative": DEMOGRAPHICS}
GROUP_NCOLS = {"Vital signs": 4, "Laboratory values": 6, "Demographic / administrative": 3}


def new_grid(n_panels, ncols, cell=(3.3, 2.7), **kwargs):
    '''A grid of subplots with room for n_panels panels (the unused ones are hidden).'''
    nrows = ceil(n_panels / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * cell[0], nrows * cell[1]), **kwargs)
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[n_panels:]:
        ax.set_visible(False)
    return fig, axes[:n_panels]


def show_no_data(ax, message="no data"):
    ax.text(0.5, 0.5, message, ha="center", va="center", transform=ax.transAxes, color="gray", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])


def compare_features(selections, features, ncols):
    '''One panel per feature. selections = [(label, dataframe, row_mask or None), ...]'''
    fig, axes = new_grid(len(features), ncols)
    labels = [label for label, _, _ in selections]

    for ax, feature in zip(axes, features):
        values = []
        for _, df, mask in selections:
            v = df[feature].to_numpy()
            v = v if mask is None else v[mask]
            values.append(v[~np.isnan(v)])    # only real measurements, gaps are not values

        if all(len(v) == 0 for v in values):
            show_no_data(ax)
        elif feature in BINARY:
            ax.bar(labels, [v.mean() * 100 if len(v) else np.nan for v in values],
                   color=[COLORS[label] for label in labels])
            ax.set_ylim(0, 100)
            ax.set_ylabel("% of rows = 1", fontsize=8)
        else:
            # outliers hidden: a few extreme lab values would squash every box into a line
            positions = [i + 1 for i, v in enumerate(values) if len(v)]
            boxes = ax.boxplot([v for v in values if len(v)], positions=positions, widths=0.6,
                               showfliers=False, patch_artist=True, medianprops={"color": "black"})
            for box, position in zip(boxes["boxes"], positions):
                box.set_facecolor(COLORS[labels[position - 1]])
            ax.set_xticks(range(1, len(labels) + 1))
            ax.set_xticklabels(labels, fontsize=8)
            ax.set_xlim(0.4, len(labels) + 0.6)
            ax.set_ylabel("Measured value", fontsize=8)
        ax.set_title(feature)
    return fig


def compare_all_features(selections, title):
    '''Vital signs, lab values and demographics, one figure each (40 panels in one figure is unreadable).'''
    for group_name, features in FEATURE_GROUPS.items():
        fig = compare_features(selections, features, ncols=GROUP_NCOLS[group_name])
        fig.suptitle(f"{group_name} — {title}", fontsize=13)
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        plt.show()

## 1. Class balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (hospital, df) in zip(axes, HOSPITALS.items()):
    status = s.patient_status(df)            # one value per patient: 1 = septic at some point
    ax.bar(["Non-septic", "Septic"], [(status == 0).sum(), (status == 1).sum()])
    ax.set_title(hospital)
    ax.set_ylabel("Number of Patients")

plt.tight_layout()
plt.show()

for hospital, df in HOSPITALS.items():
    status = s.patient_status(df)
    total_patients, septic = len(status), int(status.sum())
    total_hours, positive_hours = len(df), int(df["SepsisLabel"].sum())
    non_septic, negative_hours = total_patients - septic, total_hours - positive_hours

    print(f"\n{hospital}")
    print(f"Total patients: {total_patients:,}")
    print(f"Septic patients: {septic:,} ({septic / total_patients * 100:.2f}%)")
    print(f"Non-septic patients: {non_septic:,} ({non_septic / total_patients * 100:.2f}%)")
    print(f"\nTotal hourly observations: {total_hours:,}")
    print(f"Positive hours: {positive_hours:,} ({positive_hours / total_hours * 100:.2f}%)")
    print(f"Negative hours: {negative_hours:,} ({negative_hours / total_hours * 100:.2f}%)")
    print(f"\nAverage recorded ICU hours per patient: {total_hours / total_patients:.2f}")

The results show that both hospitals have a highly imbalanced dataset, with only 8.80% of patients in Hospital A and 5.71% in Hospital B developing sepsis. This imbalance is even more noticeable at the hourly level, where positive labels represent only 2.17% and 1.41% of observations, respectively. Since our models will make predictions every hour, this means that a model could achieve high accuracy by mostly predicting the negative class, while failing to identify sepsis. Therefore, evaluation metrics such as AUPRC, recall, and clinical utility will be particularly important. Another observation is that sepsis prevalence is lower in Hospital B, which could affect model performance when training on Hospital A and testing on Hospital B. On the other hand, the average recording duration is similar in both hospitals, at approximately 38 hours. Overall, these findings highlight the need to account for class imbalance and carefully evaluate how well our models generalize across hospitals.

## 2. Missingness

In [ ]:
# % of hours in which each feature is NOT measured
missing_data = pd.DataFrame({hospital: df[ALL_FEATURES].isna().mean() * 100
                             for hospital, df in HOSPITALS.items()})

plt.figure(figsize=(8, 12))
sns.heatmap(missing_data, annot=True, fmt=".1f", cmap="Reds", vmin=0, vmax=100)
plt.title("Missing Values (%) in Hospitals A and B")
plt.xlabel("Hospital")
plt.ylabel("Clinical Features")
plt.tight_layout()
plt.show()

This is a heatmap showing the percentages of missing values in each hospital. We can see that most laboratory measures, like Bilirubin_total, TroponinI, Lactate, etc., have missing value percentages at almost 90% or higher. On the other hand, vital signs have fewer missing values except for temperature which is missing in around 66% of observations.

One good thing is that the missingness percentages follow almost the same pattern in Hospitals A and B. However, there are some differences, such as DBP, which has 48.1% missing values in Hospital A compared to only 13.9% in Hospital B

One question is: If a person has sepsis, would that person have fewer missing values because laboratory tests are asked to be taken when a person shows signs of sepsis? If this is the case, missingness itself could provide useful information for predicting sepsis.

Another question is: Do different hospitals have different protocols for such cases? This can make missingness an issue because lab values being present may be highly dependent on hospital protocols. This could affect how well a model trained on Hospital A performs on Hospital B, especially if the model learns patterns related to how frequently measurements are taken.

Therefore, it would be interesting to compare missingness between septic and non-septic patients and investigate whether these differences appear before sepsis onset. Also, having a high percentage of missing values does not necessarily mean that a feature is not useful for predicting sepsis.

In [ ]:
# Same %, but separately for patients who do and do not become septic.
# If doctors order more tests when they suspect sepsis, the gaps themselves carry information.
fig, axes = plt.subplots(1, 2, figsize=(16, 14), sharey=True)

for ax, (hospital, df) in zip(axes, HOSPITALS.items()):
    comparison = pd.DataFrame({status: df.loc[df["PatientStatus"] == status, ALL_FEATURES].isna().mean() * 100
                               for status in ["Septic", "Non-septic"]})
    comparison.plot(kind="barh", ax=ax)
    ax.set_title(hospital)
    ax.set_xlabel("Missing Values (%)")
    ax.set_xlim(0, 100)
    ax.legend()

plt.suptitle("Missingness in Septic vs. Non-Septic Patients")
plt.tight_layout()
plt.show()

From Hospital A, we can see that, in fact, missingness is slightly affected by whether a person develops sepsis or not. Specifically, almost all lab values are present more frequently when a person develops sepsis, with some exceptions, such as lactate. However, missingness in these values is still very high for both groups across almost all laboratory measurements.

One thing that is noticeable is that patients who do not develop sepsis have the unit numbers (Unit1 and Unit2) present more frequently. Could that be a coincidence? One possible explanation is that patients are assigned to different ICU units depending on their clinical conditions. However this does not explain why the unit information itself is missing more frequently among septic patients. We also see that patients who develop sepsis have their blood pressures taken more often, particularly MAP and DBP.

We observe almost the same pattern in Hospital B, with some differences. In Hospital B, we can see that EtCO2 is measured much more frequently when a person develops sepsis than when a person does not. However, in Hospital A, this measurement is not taken at all. This means that we would need to remove this feature from our main model, since we are training on Hospital A and testing on Hospital B. However, this could lead to the loss of a potentially valuable feature, especially since its availability seems to be related to sepsis status in Hospital B.

The missingness patterns are generally similar across both hospitals, with some noticeable differences, particularly EtCO2. Therefore, although hospital practices might affect the availability of certain measurements, the results do not provide enough evidence to conclude that missingness is strongly dependent on hospital protocols

## 3. Feature distributions

In [ ]:
# Septic vs. non-septic patients, all hours of the stay
for hospital, df in HOSPITALS.items():
    selections = [(status, df, (df["PatientStatus"] == status).to_numpy())
                  for status in ["Non-septic", "Septic"]]
    compare_all_features(selections, f"Septic vs. Non-Septic Patients ({hospital})")

Some features we expect to differ, and why:

- **Heart rate (HR):** sepsis can cause tachycardia (increased heart rate).
- **Mean arterial pressure (MAP):** sepsis can cause hypotension due to circulatory dysfunction.
- **Respiratory rate (Resp):** an increased respiratory rate can be an early sign of physiological deterioration.
- **Temperature (Temp):** fever or hypothermia can indicate infection.
- **Lactate:** elevated lactate can indicate impaired tissue perfusion or metabolic stress.
- **White blood cell count (WBC):** abnormal values can reflect an inflammatory response.

We can see that these are verified. However, here we are finding these values for when sepsis label is 1 as well as before that for septic patients... What if we saw that data only before the patient develops sepsis since our aim is to predict sepsis?

In [ ]:
# Non-septic = patients who never develop sepsis
# Pre-sepsis = hours of septic patients BEFORE onset  <- the hours a model must learn from
# Sepsis     = hours of septic patients from onset on
for hospital, df in HOSPITALS.items():
    selections = [(phase, df, (df["Phase"] == phase).to_numpy())
                  for phase in ["Non-septic", "Pre-sepsis", "Sepsis"]]
    compare_all_features(selections, f"Before and After Sepsis ({hospital})")

We can see that the means and standard deviations do differ pre-sepsis and resemble the sepsis case more that the non-sepsis case... which could mean that these features could help us in predicting sepsis.

In [ ]:
# Pairwise correlation; pandas skips the missing values pair by pair
for hospital, df in HOSPITALS.items():
    plt.figure(figsize=(16, 14))
    sns.heatmap(df[ALL_FEATURES].corr(), annot=True, fmt=".1f", annot_kws={"size": 6},
                cmap="coolwarm", vmin=-1, vmax=1, square=True, cbar_kws={"shrink": 0.7})
    plt.title(f"Correlation Between All Features — {hospital}")
    plt.tight_layout()
    plt.show()

Why do we want to see the correlation between features?
Seeing how features are correlated would help us see if any features are repetitive where we can reduce the number of features without losing much predictive value... one very interesting thing would be to show the correlation between features and sepsis.

In [ ]:
# Correlation of every feature with the label itself
comparison = pd.DataFrame({
    hospital: df[ALL_FEATURES + ["SepsisLabel"]].corr()["SepsisLabel"].drop("SepsisLabel")
    for hospital, df in HOSPITALS.items()
})

plt.figure(figsize=(8, 12))
sns.heatmap(comparison, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Between SepsisLabel and Features")
plt.tight_layout()
plt.show()

In [ ]:
# We train on A and test on B, so any feature that is distributed differently in B is a warning sign
selections = [(hospital, df, None) for hospital, df in HOSPITALS.items()]
compare_all_features(selections, "Hospital A vs. Hospital B (all hourly observations)")

We can see that the mean values are almost similar in both hospitals except for MAP.

## 4. The 24 hours before sepsis onset

Only septic patients here. The red dashed line is the first positive label (6 hours before onset), the black one is onset.

In [ ]:
HOURS = np.arange(-24, 1)     # hours relative to onset
MIN_N = 10                    # draw an hour only if at least 10 septic patients have a value then (fewer = noise)


def before_onset(table, df):
    '''Rows of `table` from septic patients' hours -24 ... 0, grouped by hour relative to onset.'''
    window = df["Septic"] & df["HoursToOnset"].between(-24, 0)
    return table[window].groupby(df.loc[window, "HoursToOnset"].astype(int).to_numpy())


for hospital, df in HOSPITALS.items():
    # mean of True/False = share measured; rows = hours, columns = features
    rate = before_onset(df[CLINICAL].notna(), df).mean().reindex(HOURS) * 100

    fig, (ax_vitals, ax_labs) = plt.subplots(2, 1, figsize=(12, 12),
                                             gridspec_kw={"height_ratios": [len(VITALS), len(LABS)]})

    for ax, features in ((ax_vitals, VITALS), (ax_labs, LABS)):
        values = rate[features].T
        top = np.nanmax(values.to_numpy()) if values.notna().any().any() else 1
        sns.heatmap(values, ax=ax, cmap="viridis", vmin=0,
                    vmax=100 if features is VITALS else max(top, 1),     # labs are rare: own color scale, or all would be dark
                    cbar_kws={"label": "% of septic patients with a measurement"})
        ax.axvline(list(HOURS).index(-6) + 0.5, color="red", linestyle="--")
        ax.axvline(list(HOURS).index(0) + 0.5, color="black", linestyle="--")
        ax.set_xlabel("Hours relative to sepsis onset")

    ax_vitals.set_title(f"Vital signs: measurement rate before sepsis onset — {hospital}")
    ax_labs.set_title(f"Laboratory values: measurement rate before sepsis onset — {hospital}")
    plt.tight_layout()
    plt.show()

In [ ]:
def median_iqr_before_onset(df, features):
    grouped = before_onset(df[features], df)
    enough = grouped.count().reindex(HOURS) >= MIN_N
    return {"median": grouped.median().reindex(HOURS).where(enough),
            "q1": grouped.quantile(0.25).reindex(HOURS).where(enough),
            "q3": grouped.quantile(0.75).reindex(HOURS).where(enough)}


def plot_before_onset(curves, group_name, ylabel, title):
    '''One panel per feature, Hospital A and B in the same panel. curves[hospital][feature] -> values per hour.'''
    features = FEATURE_GROUPS[group_name]
    fig, axes = new_grid(len(features), GROUP_NCOLS[group_name])

    legend_ax = None
    for ax, feature in zip(axes, features):
        ax.set_title(feature)
        if all(curves[hospital]["median"][feature].isna().all() for hospital in HOSPITALS):
            show_no_data(ax, f"not enough data\n(fewer than {MIN_N} values per hour)")
            continue

        for hospital in HOSPITALS:
            stats = curves[hospital]
            ax.plot(HOURS, stats["median"][feature], color=COLORS[hospital], label=hospital)
            if "q1" in stats:     # the IQR band; the variability plot only has a median
                ax.fill_between(HOURS, stats["q1"][feature], stats["q3"][feature], color=COLORS[hospital], alpha=0.15)

        ax.axvline(-6, color="red", linestyle="--", linewidth=1)
        ax.axvline(0, color="black", linestyle="--", linewidth=1)
        ax.set_xlabel("Hours relative to onset", fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.grid(alpha=0.3)
        legend_ax = legend_ax or ax

    if legend_ax is not None:
        legend_ax.legend(fontsize=8)
    fig.suptitle(f"{title} (red: first positive label, black: onset)", fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    plt.show()


trajectories = {hospital: median_iqr_before_onset(df, CLINICAL) for hospital, df in HOSPITALS.items()}
for group_name in ["Vital signs", "Laboratory values"]:
    plot_before_onset(trajectories, group_name, "Measured value", f"{group_name} Before Sepsis Onset: median and IQR")

In [ ]:
# A few individual patients: averages can hide what one stay really looks like
selected = {}

for hospital, df in HOSPITALS.items():
    septic_patient = df.groupby("PatientID")["Septic"].first()
    np.random.seed(42)      # same patients every run
    selected[hospital] = {
        "Septic": list(np.random.choice(septic_patient[septic_patient].index.to_numpy(), 10, replace=False)),
        "Non-septic": list(np.random.choice(septic_patient[~septic_patient].index.to_numpy(), 10, replace=False)),
    }


def plot_patient_timeline(df, patient_id, hospital):
    patient = df[df["PatientID"] == patient_id]
    septic = bool(patient["Septic"].iloc[0])

    fig, axes = new_grid(len(CLINICAL), ncols=6, cell=(3.0, 2.1))
    for ax, feature in zip(axes, CLINICAL):
        values = patient[["Hour", feature]].dropna()   # dots = real measurements only
        if values.empty:
            show_no_data(ax)
        else:
            ax.plot(values["Hour"], values[feature], marker="o", markersize=3, linewidth=1)

        if septic:
            onset = patient["OnsetHour"].iloc[0]
            ax.axvline(onset - 6, color="red", linestyle="--", linewidth=1)
            ax.axvline(onset, color="black", linestyle="--", linewidth=1)
        ax.set_title(feature, fontsize=9)

    status = "septic" if septic else "non-septic"
    fig.suptitle(f"{hospital}: {status} patient {patient_id} (hours since first record; "
                 f"red: first positive label, black: onset)", fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    plt.show()


# One patient of each kind per hospital; any other ID from selected[hospital][...] works too
for hospital, df in HOSPITALS.items():
    for status in ["Septic", "Non-septic"]:
        plot_patient_timeline(df, selected[hospital][status][0], hospital)

In [ ]:
# Variability = rolling standard deviation over the previous `window` hours.
# Vitals are measured about hourly: 6 h with at least 3 values.
# Labs come about once a day, so that rule would almost never be met: 24 h with at least 2 values instead.
VARIABILITY_SETTINGS = {
    "Vital signs": dict(features=VITALS, window=6, min_obs=3),
    "Laboratory values": dict(features=LABS, window=24, min_obs=2),
}


def variability_before_onset(df, features, window, min_obs):
    septic = df.loc[df["Septic"], ["PatientID", "Hour", "HoursToOnset"] + features]
    rolling_std = (septic.groupby("PatientID")[features]
                         .rolling(window, min_periods=min_obs).std()
                         .droplevel(0)
                         .reindex(septic.index))

    # only windows that lie completely inside the stay
    use = septic["HoursToOnset"].between(-24, 0) & (septic["Hour"] >= window - 1)
    grouped = rolling_std[use].groupby(septic.loc[use, "HoursToOnset"].astype(int).to_numpy())
    enough = grouped.count().reindex(HOURS) >= MIN_N
    return {"median": grouped.median().reindex(HOURS).where(enough)}


for group, settings in VARIABILITY_SETTINGS.items():
    curves = {hospital: variability_before_onset(df, **settings) for hospital, df in HOSPITALS.items()}
    plot_before_onset(curves, group, f"{settings['window']}-Hour Standard Deviation",
                      f"Temporal Variability of {group} Before Sepsis Onset (median of the {settings['window']}-hour rolling SD)")

These graphs show how the variability of clinical measurements changes during the 24 hours before sepsis onset in both hospitals. We can see that for most vital signs the standard deviation does not change that much, however some features start showing an increase as sepsis onset approaches. For example HR variability increases slightly in both hospitals while MAP and DBP increase more in Hospital B. One thing that is interesting is that some features have different variability between hospitals, like O2Sat which is more variable in Hospital B while respiratory rate is more variable in Hospital A.

For laboratory measurements the differences between hospitals are much larger. We can see that features like glucose, WBC, Hct and Hgb have much higher variability in Hospital B. Also some features like glucose, creatinine and fibrinogen seem to become more variable as sepsis onset approaches. Could this mean that the changes in these features could help us predict sepsis earlier?

However we need to remember that most laboratory measurements have very high missingness percentages, which means that these standard deviations might be calculated using only a few patients. This could explain why some graphs are almost flat while others have sudden changes. So we cannot really know if these differences are related to the patients themselves or simply because of how often the measurements are taken.

Overall we can see that looking at how the features change over time might be useful instead of only looking at their values at a certain hour. However we are only looking at patients who develop sepsis here, so we still need to compare these changes with patients who do not develop sepsis to see if these patterns are actually related to sepsis.

## 5. Time in the ICU

In [ ]:
# Does the share of positive labels change with time in the ICU? (6-hour bins)
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax1, (hospital, df) in zip(axes, HOSPITALS.items()):
    hour_group = (df["ICULOS"] // 6) * 6
    sepsis_rate = df.groupby(hour_group)["SepsisLabel"].mean() * 100
    observations = df.groupby(hour_group).size()     # late bins hold few patients, so read them with care

    ax1.plot(sepsis_rate.index, sepsis_rate.values, marker="o")
    ax1.set_xlabel("Hours in ICU")
    ax1.set_ylabel("Positive Sepsis Labels (%)")
    ax1.set_title(hospital)
    ax1.grid(alpha=0.3)

    ax2 = ax1.twinx()
    ax2.bar(observations.index, observations.values, width=5, alpha=0.2, color="gray")
    ax2.set_ylabel("Number of Hourly Observations")

plt.suptitle("Sepsis Label Rate by ICU Stay Duration")
plt.tight_layout()
plt.show()

In [ ]:
# Share of each group's features that are present in a row, averaged per 6-hour bin
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, (hospital, df) in zip(axes, HOSPITALS.items()):
    hour_group = (df["ICULOS"] // 6) * 6
    per_row = pd.DataFrame({name: df[columns].notna().mean(axis=1) * 100
                            for name, columns in FEATURE_GROUPS.items()})
    table = per_row.groupby(hour_group).mean()

    for group in table.columns:
        ax.plot(table.index, table[group], marker="o", label=group)
    ax.set_xlabel("Hours in ICU")
    ax.set_ylabel("Average Feature Availability (%)")
    ax.set_title(hospital)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Feature Availability by Clinical Group")
plt.tight_layout()
plt.show()

From these graphs we can see that vital signs are much more available than laboratory measurements in both hospitals, which is something we already noticed from the missingness heatmap. Laboratory measurements have a very low availability, around 5% in Hospital A and even less in Hospital B. One thing that is interesting is that their availability does not really increase as the patient spends more time in the ICU, it stays almost the same throughout the stay. This could mean that laboratory tests are not necessarily taken more frequently just because a patient has been in the ICU for longer.

For vital signs, we can see that their availability is around 65-75% in both hospitals, with Hospital B having slightly higher availability for most of the ICU stay. We also notice that there are some fluctuations but no major changes over time. However towards the end, especially in Hospital B, the availability starts decreasing. This could be because fewer patients remain in the dataset at these later hours, so the results might be less reliable.

Another thing we can notice is that demographic features are more available in Hospital B than in Hospital A, around 90% compared to around 80%. This is probably related to the differences in missingness of Unit1 and Unit2 that we noticed earlier, since the other demographic features are almost always present.

Overall, we can see that the availability of features does not change much with ICU duration, and that the main difference is between the feature groups themselves rather than how long the patient stays in the ICU. This makes us wonder whether laboratory measurements, despite being so rarely available, could still improve sepsis prediction compared to using vital signs alone. It also shows why comparing different feature groups is important for our project.